# CBB26 — Plant phylogeny using chloroplast rbcL

Maximum-likelihood inference for five versioned chloroplast records. The primary analysis and exploratory site subsets are reproduced in an isolated output directory.

## 1. Dependencies

In [ ]:
from pathlib import Path
import hashlib
import importlib.util
import json
import shutil
import subprocess
import sys
import tempfile
from zipfile import ZipFile

def run(args, cwd=None):
    subprocess.run([str(arg) for arg in args], cwd=cwd, check=True)

packages = [name for name, module in [('biopython', 'Bio'), ('matplotlib', 'matplotlib'), ('pandas', 'pandas')] if importlib.util.find_spec(module) is None]
if packages:
    run([sys.executable, '-m', 'pip', 'install', *packages])

if not all([shutil.which('mafft'), shutil.which('seqkit'), shutil.which('iqtree2') or shutil.which('iqtree')]):
    run(['apt-get', 'update', '-qq'])
    run(['apt-get', 'install', '-y', 'mafft', 'seqkit', 'iqtree'])

## 2. Archived inputs and integrity

In [ ]:
# Locate the supplied project directory or archive in the working or mounted input directories.
roots = [Path.cwd(), *Path('/').glob('*/input')]
folders = []
archives = []
for root in roots:
    if (root / 'metadata/primary_checksums.sha256').is_file():
        folders.append(root)
    folders.extend(p.parent.parent for p in root.rglob('primary_checksums.sha256') if p.parent.name == 'metadata')
    archives.extend(root.rglob('CBB26_Project3_Submission_Final.zip'))

project = Path(tempfile.mkdtemp(prefix='rbcl_', dir=Path.cwd().parent if (Path.cwd() / 'metadata/primary_checksums.sha256').is_file() else Path.cwd())) / 'Project3_rbcL'
if folders:
    shutil.copytree(folders[0], project)
elif archives:
    with ZipFile(archives[0]) as archive:
        archive.extractall(project.parent)
else:
    raise FileNotFoundError('The project directory or submission archive is required.')

for line in (project / 'metadata/primary_checksums.sha256').read_text().splitlines():
    expected, relative = line.split(maxsplit=1)
    actual = hashlib.sha256((project / relative).read_bytes()).hexdigest()
    if actual != expected:
        raise ValueError(f'Checksum mismatch: {relative}')

# Isolate new outputs from the archived primary analysis.
for folder in ('results', 'logs', 'figures'):
    shutil.rmtree(project / folder)
    (project / folder).mkdir()
# The archived report and its manifest describe the original run only.
shutil.rmtree(project / 'docs')
for name in ('artifact_checksums.sha256', 'primary_checksums.sha256'):
    (project / 'metadata' / name).unlink()
print(project)

## 3. Primary analysis

In [ ]:
run([sys.executable, 'scripts/pipeline.py', 'all', '--offline', '--threads', '2', '--seed', '42'], cwd=project)

## 4. Primary results

In [ ]:
import pandas as pd
from IPython.display import display, Image

for name in ('sequence_qc.tsv', 'alignment_per_taxon.tsv', 'branch_support.tsv'):
    display(pd.read_csv(project / 'results' / name, sep='\t'))
for name in ('alignment_qc.json', 'tree_evidence.json', 'site_diagnostics.json'):
    print(json.dumps(json.loads((project / 'results' / name).read_text()), indent=2))
display(Image(filename=str(project / 'figures/rbcl_tree.png')))

## 5. Exploratory site subsets

In [ ]:
run([sys.executable, 'scripts/sensitivity.py', '--threads', '1', '--seed', '42', '--output', 'results/sensitivity'], cwd=project)
summary = json.loads((project / 'results/sensitivity/summary.json').read_text())
rows = []
for name, result in summary.items():
    splits = result['observed_splits_ufboot_pct']
    rows.append({'subset': name, 'columns': result['columns'], 'model': result['model'],
                 'family_split': splits.get('Arabidopsis_thaliana; Brassica_rapa', 'Not recovered'),
                 'wheat_maize': splits.get('Triticum_aestivum; Zea_mays', 'Not recovered'),
                 'rice_maize': splits.get('Oryza_sativa; Zea_mays', 'Not recovered')})
display(pd.DataFrame(rows))
print(f'Analysis outputs: {project / "results"}')